# 🥇 XAUUSD Neural Lab - Advanced Bi-LSTM Training

Jupyter Notebook ini didesain untuk dijalankan di **Kaggle** atau **Google Colab**.
Model ini adalah arsitektur **3-Layer Bidirectional LSTM** yang akan mempelajari pola pergerakan harga Emas (XAUUSD) digali dari 22+ *Technical Features*, serta data korelasi makro-ekonomi seperti **DXY (Dollar Index)** dan **US10Y (Bond Yields)**.

### 🚀 Cara Penggunaan:
1. Jalankan semua cell secara berurutan.
2. Di bagian Dataset, pastikan Anda telah mengunggah file CSV historis XAUUSD yang berisi OHLCV, DXY, US10Y, dsb.
3. Setelah *training* selesai, script ini akan menghasilkan file `lstm_weights_v2.json`.
4. Copy isi file JSON tersebut dan gantikan *existing weights* di web app (file `src/lib/lstm-weights.ts`).

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import json
import datetime
import os

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Load & Preprocess Dataset
Diasumsikan Anda memiliki file `xauusd_merged_1h.csv` dengan kolom OHLCV murni ditambah:
`dxy_close` (USD Index), `us10y_close` (Bond Yield 10 Year), `cot_index` (Commitment of Traders), `is_nfp_week` (Non-Farm Payroll Flag).

In [ ]:
# DUMMY DATA GENERATOR (Hapus block ini dan gunakan pandas read_csv jika data asli sudah ada)
def generate_dummy_data(rows=10000):
    np.random.seed(42)
    dates = pd.date_range(start='2020-01-01', periods=rows, freq='1H')
    close = np.cumsum(np.random.randn(rows)) + 2000
    df = pd.DataFrame({
        'time': dates,
        'open': close + np.random.randn(rows) * 0.5,
        'high': close + np.abs(np.random.randn(rows)),
        'low': close - np.abs(np.random.randn(rows)),
        'close': close,
        'volume': np.random.randint(1000, 10000, rows),
        'dxy_close': 100 + np.cumsum(np.random.randn(rows) * 0.1),  # Dollar index
        'us10y_close': 4.0 + np.cumsum(np.random.randn(rows) * 0.01) # Yields
    })
    return df

# df = pd.read_csv('xauusd_merged_1h.csv') # UNCOMMENT UNTUK LOAD CSV KAGGLE ANDA
df = generate_dummy_data() 

print("Dataset Shape:", df.shape)
df.head()

## 2. Feature Engineering (Technical + Macro)
Menghitung The 22-Feature Extractor ditambah MACRO parameters.

In [ ]:
def add_features(df):
    # 1. Moving Averages
    df['ema_20'] = df['close'].ewm(span=20, adjust=False).mean()
    df['ema_50'] = df['close'].ewm(span=50, adjust=False).mean()
    df['ema_200'] = df['close'].ewm(span=200, adjust=False).mean()
    
    # 2. Normalized Close
    df['norm_close'] = (df['close'] - df['ema_20']) / (df['ema_20'] * 0.01)
    
    # 3. Target Variable (3-Class: UP, DOWN, NEUTRAL)
    # Prediksi 1 candle ke depan. Jika naik > 0.05%, UP. Turun < -0.05%, DOWN. Sisanya NEUTRAL.
    future_return = df['close'].shift(-1) / df['close'] - 1
    threshold = 0.0005 # 0.05%
    
    conditions = [
        (future_return > threshold),
        (future_return < -threshold)
    ]
    choices = [0, 1] # 0: UP, 1: DOWN
    df['target'] = np.select(conditions, choices, default=2) # 2: NEUTRAL
    
    # 4. Fibonacci Retracement (Daily High/Low)
    rolling_high = df['high'].rolling(window=24).max()
    rolling_low = df['low'].rolling(window=24).min()
    diff = rolling_high - rolling_low
    df['fib_618'] = rolling_high - diff * 0.618
    df['fib_dist'] = (df['close'] - df['fib_618']) / df['close'] * 100
    
    # 5. Correlation Features (DXY & US10Y Momentum)
    if 'dxy_close' in df.columns:
        df['dxy_momentum'] = df['dxy_close'].pct_change(10) * 100
    else:
        df['dxy_momentum'] = 0
        
    df.dropna(inplace=True)
    return df

df_features = add_features(df)
print("Classes distribution:\n", df_features['target'].value_counts(normalize=True))

## 3. Create Sequences for LSTM
LSTM membutuhkan data tiga dimensi: `[samples, lookback, features]`

In [ ]:
LOOKBACK = 60

# Pilih fitur yang akan dimasukkan ke neural network (sesuaikan dengan neural-lab-features.ts)
feature_cols = ['norm_close', 'fib_dist', 'dxy_momentum', 'volume'] 
# WARNING: Di versi asli, array features akan berisi lebih dari 22 kolom.

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_features[feature_cols])
targets = pd.get_dummies(df_features['target']).values # One-hot encoding untuk 3 kelas

X, y = [], []
for i in range(LOOKBACK, len(scaled_data)):
    X.append(scaled_data[i-LOOKBACK:i])
    y.append(targets[i])

X, y = np.array(X), np.array(y)

# Train-Test Split (80% Train, 20% Test, secara kronologis)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

## 4. Build Bi-LSTM Architecture
Arsitektur VVIP: 3-Layer Bidirectional LSTM dengan Dropout untuk mencegah Overfitting.

In [ ]:
model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(LOOKBACK, len(feature_cols))),
    Dropout(0.2),
    
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    
    Bidirectional(LSTM(32, return_sequences=False)),
    Dropout(0.2),
    
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(3, activation='softmax') # 3 Classes: UP, DOWN, NEUTRAL
])

optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

## 5. Training Phase

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## 6. Evaluation

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy Over Epochs')
plt.legend()
plt.show()

loss, acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {acc*100:.2f}%")

## 7. Export Weights for Next.js (TypeScript)
Model LSTM JavaScript di Arra7 membaca model dalam bentuk matriks array.

In [ ]:
def export_weights_to_json(model, filename='lstm_weights_v2.json'):
    weights_dict = {}
    for layer in model.layers:
        layer_weights = layer.get_weights()
        if len(layer_weights) > 0:
            # Convert numpy arrays to lists
            weights_dict[layer.name] = [w.tolist() for w in layer_weights]
    
    # Tambahkan Metadata
    total_params = model.count_params()
    export_data = {
        'metadata': {
            'architecture': 'Bi-LSTM 3-Layer (VVIP Edition)',
            'biLstmUnits': [128, 64, 32],
            'denseUnits': [64, 3],
            'totalParams': total_params,
            'accuracy': float(acc),
            'trainedAt': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC"),
            'lookback': LOOKBACK,
            'epochs': len(history.history['loss']),
            'features': len(feature_cols)
        },
        'weights': weights_dict
    }
    
    with open(filename, 'w') as f:
        json.dump(export_data, f)
        
    print(f"✅ Weights successfully exported to {filename}")
    print(f"File size: {os.path.getsize(filename) / (1024*1024):.2f} MB")

export_weights_to_json(model)